# Modul 06 · Notebook 02 — NVIDIA NIM (Generator di Cloud)

Model NVIDIA terkini seperti **Nemotron 3** butuh banyak memori — versi besarnya **tidak muat** di T4. Solusinya: jangan jalankan di GPU-mu, **panggil dari cloud**.

**NVIDIA NIM** (*NVIDIA Inference Microservices*) menyajikan model lewat API **kompatibel-OpenAI** di `https://integrate.api.nvidia.com/v1`. Kode yang sama untuk OpenAI bisa langsung memanggil model NVIDIA — **tanpa GPU di sisimu**. Kita pakai **`nvidia/nemotron-3-nano-30b-a3b`** (NVIDIA-native, MoE 30B/~3B aktif).

> 🔑 Ambil API key gratis di [build.nvidia.com](https://build.nvidia.com), tambahkan ke **Colab Secrets** (ikon 🔑) dengan nama **`NVIDIA_API_KEY`**, aktifkan akses notebook.

In [ ]:
# Instal openai + ambil helper bersama (nvidia_utils.py)
!pip -q install openai
import os
if not os.path.exists("nvidia_utils.py"):
    !wget -q https://raw.githubusercontent.com/chmdznr/navasena-gen-ml-course/module06-rework/06_nvidia_ecosystem/tools/nvidia_utils.py

In [ ]:
# Ambil key dari Colab Secrets (JANGAN pernah menaruh key langsung di kode)
import os
from google.colab import userdata
os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")
print("Key termuat:", bool(os.environ.get("NVIDIA_API_KEY")))

## Panggil model NVIDIA di cloud

Nemotron 3 adalah model **reasoning**; di NIM, mode berpikir dimatikan lewat parameter API — helper **`nim_chat()`** sudah mengurusnya (output cepat & bersih, tanpa jejak `<think>`).

In [ ]:
from nvidia_utils import nim_client, nim_chat
client = nim_client()                          # default -> https://integrate.api.nvidia.com/v1
MODEL = "nvidia/nemotron-3-nano-30b-a3b"

print(nim_chat(client, MODEL, messages=[
    {"role": "user", "content": "Apa itu retrieval-augmented generation (RAG)? Jawab singkat dalam Bahasa Indonesia."}]))

## Kenapa "kompatibel-OpenAI" itu penting?

Kode aplikasimu jadi **portabel**: Ollama (lokal), vLLM, NVIDIA NIM — semua bicara protokol sama. Pindah target cukup ganti `base_url`. Ini pola yang sama seperti Modul 04 (deployment SLM): satu kode klien, banyak backend.

In [ ]:
# Satu key, dua peran: model yang sama bisa jadi "juri" (LLM-as-judge)
jawaban = "RAG menggabungkan pencarian dokumen dengan LLM untuk menjawab berdasarkan sumber."
skor = nim_chat(client, MODEL, max_tokens=8, messages=[
    {"role": "user", "content": f"Beri skor 1-5 untuk kejelasan jawaban ini (balas ANGKA saja): '{jawaban}'"}])
print("Skor juri:", skor)

## 🧪 Latihan

1. Ganti `MODEL` ke model NIM lain dari [build.nvidia.com](https://build.nvidia.com) — bandingkan gaya jawabannya.
2. Tambah argumen `temperature=0.7` pada `nim_chat(...)` — apakah jawaban jadi lebih bervariasi?
3. Pikirkan: kenapa LLM-as-judge berguna untuk **mengukur** kualitas (ingat Modul 04 evaluation)? Konsep ini kembali di **nb04** sebagai *self-check rail*.